# Herschrijven — trainingen naar de nieuwe stijl

Draai de pijplijn stap voor stap, met de mens als poort op de plek waar dat telt.

**Secties 1, 2, 4 en 10 doen géén API-calls**, en in sectie 3 alleen de goedkope classificatie
van de vrije-tekst-aantekeningen (Haiku). Je leest in, joint, normaliseert de besluiten en
inspecteert wat het model straks precies te zien krijgt — vóórdat de dure schrijfcalls lopen.

**Voor je begint:**
1. `pip install -r requirements.txt`
2. Zet je API-key in een `.env` naast dit notebook.
3. Twee inputbestanden: het **scoresheet** (scorer-velden + de handmatig ingevulde kolommen
   `actie_besluit`, `kern_reviewer`, `modus_reviewer` en `guidance_reviewer`)
   en het **bronsheet** (`id` / `name` / `herschreven` / `content`).
4. `vervolgtraining.json` naast dit notebook: de catalogus van 779 trainingen
   (`product_id` / `titel` / `summary`) waaruit de Vervolgstappen worden gekozen.
5. `vervolgtrainingen_tree.json`: dezelfde trainingen, ingedeeld naar
   domein > subdomein > onderwerp. Die indeling bepaalt mee welke vervolgtrainingen op de
   shortlist komen en hoe ze gegroepeerd worden. Ontbreekt het bestand, dan valt de selectie
   terug op alleen keyword-overlap.

**Twee assen bepalen wat er met een training gebeurt.** De *modus* (`overnemen` / `stijl` /
`format` / `volledig`) zegt hoeveel van de bestaande tekst mag veranderen; de goedgekeurde
*actualiseringen* staan daar los van en worden op elk niveau doorgevoerd. Sectie 2 laat de
verdeling zien, sectie 3b vult het voorstel in.

## 1. Config

Paden en knoppen. `importlib.reload` zorgt dat je edits in de `.py`-bestanden meteen meekomen.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

import importlib
import besluiten, sjabloon, rewrite_checks, rewrite_output, rewrite_trainings
for m in (besluiten, sjabloon, rewrite_checks, rewrite_output, rewrite_trainings):
    importlib.reload(m)
bes, uit, rw = besluiten, rewrite_output, rewrite_trainings

SCORED    = "/Users/hugovandenbelt/Downloads/PRIO_TRAININGEN_SCOREN.xlsx"
SOURCE    = "/Users/hugovandenbelt/Downloads/Nieuwe lijst incl. herschreven en dagen.xlsx"
BESLUITEN = "besluiten.xlsx"
OUT_DIR   = "herschreven"

START = 0      # alleen sectie 6: 0-based rijindex in het scoresheet om mee te beginnen
N     = 1      # alleen sectie 6: aantal trainingen vanaf START (None = tot het einde)

INSPECT = 5    # alleen sectie 4/5/7/8: het ene training_id dat je onder de loep neemt

## 2. Inlezen, joinen en de populatie bekijken

Controleer eerst dat beide sheets goed binnenkomen en dat elke gescoorde training een bron heeft.

De uitsplitsing laat zien wat er automatisch herschreven wordt. Let op: een structurele
actualiteitsbreuk of `menselijke_input_nodig` stuurt een training **alleen** naar de mens als de
reviewer nog geen besluit heeft ingevuld. Heeft hij dat wel, dan is de menselijke poort al
gepasseerd en gaat de training gewoon mee.

In [ ]:
import pandas as pd

scored = rw._load_scored(SCORED)
src_by_id, cols = rw.load_source(SOURCE)
print("scoresheet:", scored.shape, "| bronkolommen:", cols)

ontbreekt = [t for t in scored["training_id"] if t not in src_by_id]
print("zonder bron:", ontbreekt or "geen")

# de reviewer-poort: staat er een ingevuld besluit tegenover de acties van deze training?
besloten = {tid for tid, lijst in bes.load_besluiten(BESLUITEN).items()
            if any(x.besluit_ruw.strip() for x in lijst)}

# AS 1 -- de modus. Zonder `modus_reviewer`/`modus_voorstel` valt hij terug op de oude
# `herschreven`-kolom, dus een sheet van vóór de schaal gedraagt zich exact als voorheen.
modus = scored.apply(
    lambda r: rw.build_briefing({k: r[k] for k in scored.columns}, {}, "").modus, axis=1)

overnemen     = modus == "overnemen"
onbruikbaar   = scored["verdict"] == "onbruikbaar"
beslissing    = ((scored["actualiteit_type"] == "structureel")
                 | scored["menselijke_input_nodig"].astype(bool))
wacht_op_mens = beslissing & ~scored["training_id"].isin(besloten)
auto          = ~(overnemen | onbruikbaar | wacht_op_mens)

print("\nAS 1 — herschrijfniveau:")
for m in rw.MODI:
    print(f"  {m:10} {int((modus == m).sum()):3d}")

print(f"""
doorgezet zonder herschrijven (overnemen) {int(overnemen.sum()):3d}
verdict onbruikbaar -> human-queue        {int(onbruikbaar.sum()):3d}
beslissing nodig, nog geen besluit        {int(wacht_op_mens.sum()):3d}
  (beslissing nodig, besluit ingevuld)    {int((beslissing & ~wacht_op_mens & ~overnemen).sum()):3d}
------------------------------------------------
gaat de auto-herschrijving in             {int(auto.sum()):3d}""")

# AS 2 -- de actualiseringen. Die lopen op ELK niveau mee, ook bij `overnemen`; daar krijgen
# ze een gerichte call die alleen de geraakte kopjes aanraakt.
met_acties = {tid for tid, lijst in bes.load_besluiten(BESLUITEN).items()
              if bes.splits(lijst)[0]}
n_overnemen_met_acties = int(scored[overnemen]["training_id"].isin(met_acties).sum())
print(f"\nAS 2 — trainingen met >=1 goedgekeurde actualisering: {len(met_acties)}"
      f"\n  waarvan op modus 'overnemen' (gerichte call, rest blijft letterlijk staan): "
      f"{n_overnemen_met_acties}")

## 3. Besluiten normaliseren — de menselijke poort

`actie_besluit` heeft een vaste *structuur* (`<nr> <vrije tekst>`) maar vrije *tekst*.
Python splitst de structuur, een klein model classificeert de aantekening als
**doen / niet / mits**, en het resultaat landt in `besluiten.xlsx`.

Draai eerst de structuurcontrole (geen API). Daarna genereer je het sheet en kijk je
**alleen de regels met `bron=llm`** na. Corrigeer wat niet klopt, zet `bron` op `handmatig`
en draai de cel opnieuw — handmatige labels worden nooit overschreven.

**In dezelfde ronde: kijk de `kern` na.** Die kolom staat in het scoresheet, met een lege
kolom `kern_reviewer` ernaast. De kern legt het **niveau** van de training vast — er is
geen apart niveau-veld — en stuurt daarmee elk kopje dat de schrijver produceert. Klopt
hij, laat `kern_reviewer` leeg. Klopt hij niet, plak hem over en pas hem aan.

Waar je op let: beschrijft de kern wat de deelnemer met het onderwerp *doet*, in de
werkwoorden van de bron? Zegt de bron "maak je kennis met" en staat er in de kern "leert
toepassen", dan is dat de correctie. En sluit de kern af met één zin over wat de training
expliciet níét doet?

Een door jou bijgestelde kern is **leidend** over de brontekst; een kern die alleen van de
scorer komt verliest juist van de brontekst en levert hooguit een melding in `notities` op.

In [3]:
fouten = bes.check_alignment(SCORED)
assert not fouten, "los eerst de uitlijnfouten op"

51 rijen, 47 met genummerde acties, 0 uitlijnfouten


In [4]:
besluiten_df = bes.write_besluiten_sheet(SCORED, BESLUITEN, verbose=False)

na_te_kijken = besluiten_df[besluiten_df["bron"] == "llm"].sort_values("confidence")
print(f"{len(besluiten_df)} besluiten; {len(na_te_kijken)} door het model geclassificeerd\n")
pd.set_option("display.max_colwidth", 70)
na_te_kijken[["training_id", "nr", "besluit_ruw", "besluit", "confidence"]]

199 besluiten; 18 door het model geclassificeerd



,training_id,nr,besluit_ruw,besluit,confidence
0,5,1,"PHP versie niet benoemen, wel relavante taalfeatures toevoegen",mits,high
41,85,1,geen specifieke versies benoemen,mits,high
38,82,2,nuanceer,mits,high
36,47,3,voeg toestandsdiagram toe als programmaonderdeel,doen,high
34,47,1,geen specifieke tools benoemen,mits,high
31,46,2,geen versienummers gebruiken,mits,high
30,46,1,beide voor zover browser variant nog relevant,mits,high
22,27,3,in inleiding is dat prima,mits,high
20,27,1,nee dat is advanced,niet,high
17,12,1,stukje over certificeringsverwijzing volledig verwijderen,mits,high


## 3b. De modus bepalen — zelfde ronde, zelfde reviewer

Hoeveel moet er aan deze training gebeuren? Vier niveaus, elk niveau mag alles wat het
niveau eronder mag:

| Modus | Wat er mag veranderen |
| --- | --- |
| `overnemen` | Niets, behalve de titel en de vervolgtitels |
| `stijl` | De formulering, naar de **actuele** schrijfregels |
| `format` | + de structuur en de ontbrekende kopjes |
| `volledig` | + de opbouw, vanaf nul uit de brontekst |

**Python kan dit niet alleen.** `rewrite_checks.py` vangt openingszinnen, lengtes, aantallen
en verboden woorden — daarmee valt te *bewijzen dat een tekst niet voldoet, nooit dat hij wél
voldoet*. Of een zin het stijlregister volgt ziet code niet. Vandaar dezelfde drietrap als bij
de besluiten: `scan_vorm()` legt een deterministische ondergrens (en stelt dus nooit
`overnemen` voor), een Haiku-call leest de actuele schrijfspec naast de bestaande tekst, en jij
beslist in `modus_reviewer`.

Kijk vooral de regels na **waar het voorstel afwijkt van de ondergrens** — daar heeft het model
iets gezien wat de checks niet zagen, of het omgekeerde. Laat `modus_reviewer` leeg als je het
met het voorstel eens bent.

De goedgekeurde actualiseringen uit sectie 3 staan hier **los** van: die worden op elk niveau
doorgevoerd, ook bij `overnemen`. Een goedgekeurde actie is een lokale toevoeging en hoort de
rest van de training niet mee te slepen.

In [ ]:
SCORED_MET_MODUS = "scoresheet_met_modus.xlsx"   # waar het voorstel naartoe wordt geschreven

# met_llm=False -> alleen de deterministische ondergrens, geen API-call en geen key nodig.
# Bruikbaar als kalibratie; als voorstel niet, want de ondergrens stelt nooit `overnemen` voor.
voorstel = rw.modus_voorstellen(SCORED, SOURCE, SCORED_MET_MODUS, met_llm=True)

afwijkend = voorstel[voorstel["modus_voorstel"] != voorstel["modus_ondergrens"]]
print(f"\n{len(afwijkend)} rijen waar het model afwijkt van de ondergrens — die eerst nalezen:")
afwijkend[["training_id", "titel", "modus_ondergrens", "modus_voorstel", "modus_reden"]]

## 4. De briefing inspecteren — nog steeds zonder API-call

Dit is wat het model straks letterlijk krijgt. Controleer drie dingen:

- de **goedgekeurde** acties staan er, mét hun voorwaarde;
- de **afgewezen** acties staan onder NIET DOEN;
- `actualiteit_specifiek` en `actualiteit_samenvatting` staan er **niet** in — dat is
  onderbouwing van de scorer, geen besluit.

In [ ]:
b = rw.build_briefing_for_id(SCORED, SOURCE, INSPECT, besluiten_path=BESLUITEN)
briefing = rw.build_writer_user(b)

print(f"{b.titel} | persona {b.persona} | {b.dagen} dagen | modus {b.modus} | "
      f"{len(b.goedgekeurd)} goedgekeurd, {len(b.afgewezen)} afgewezen\n")

# In stijl/format VERVANGT de bestaande tekst per kopje de brontekst -- zelfde content, maar
# per veld en compleet (`build_source_text` slaat setup/follow_up/summary_edudex/certification
# over). Knip het materiaalblok eraf zodat je de briefing zelf leest, niet de training.
grens = next((briefing.index(k) for k in ("OPDRACHT —", "BRONTEKST —", "HUIDIGE VERSIE —")
              if k in briefing), len(briefing))
print(briefing[:grens])

print(f"\n{'='*70}\nmateriaalblok ({'huidige versie per kopje' if b.behoudt_tekst else 'brontekst'})"
      f"\n{'='*70}\n{briefing[grens:][:1200]}")

rij = scored[scored.training_id == INSPECT].iloc[0]
for veld in ("actualiteit_specifiek", "actualiteit_samenvatting"):
    lek = str(rij[veld])[:60] in briefing
    print(f"\n{veld:28} lekt naar de schrijver: {lek}")

## 5. Eén training herschrijven

Vanaf hier lopen er API-calls. Schrijver → code-check → judge → route.

Het resultaat gaat meteen naar `herschreven/trainingen/`: `<id>.json` (lossless) en
`<id>.md` — precies het document dat je onder de cel leest, zodat je het later terugvindt.
`herschreven.xlsx` blijft hier ongemoeid; dat sheet vult de batch in sectie 6.

In [ ]:
catalog = rw.load_catalog()
boom    = rw.load_tree(catalog)   # vakgebied-indeling; stuurt de shortlist en de groepen
client  = rw.make_client()

print(f"catalogus: {len(catalog)} trainingen | in de boom ingedeeld: {len(boom['paden'])}")

res = rw.rewrite_one(client, b, catalog, boom)
print(f"status: {res.status}  {res.reden}")
print("flags:", res.flags or "geen")
print("toegepaste acties:", len(res.toegepaste_acties))

# meteen wegschrijven; de bron-content gaat mee zodat `days` overeind blijft in de JSON
bron_rij = src_by_id.get(b.training_id)
paden = rw.bewaar_training(
    OUT_DIR, res, rw.parse_content(bron_rij[cols["content"]]) if bron_rij is not None else {})
print("opgeslagen:", ", ".join(p for p in paden.values() if p), "\n")

print(uit.render_markdown(res.document, res.titel) if res.document else "(geen document)")

## 6. Batch draaien

`append=True` + `skip_existing=True`: een afgebroken run hervat zonder opnieuw te betalen.
Draai je dezelfde selectie nog eens, dan levert dat 0 nieuwe rijen op.

In [ ]:
review = rw.rewrite_file(SCORED, SOURCE, OUT_DIR,
                         besluiten_path=BESLUITEN, start=START, limit=N)

review[["training_id", "titel", "modus", "status", "reden", "thin", "n_flags", "spec_versie"]]

## 7. De output naast de bron leggen

De gegenereerde `content` heeft dezelfde sleutels als de bron, dus je kunt veld voor veld
vergelijken. Let op `days` en `certification` (ongewijzigd) en op de kop 3 in `intro`.

In [7]:
import json
from score_trainings import parse_content

with open(f"{OUT_DIR}/trainingen/{INSPECT}.json", encoding="utf-8") as f:
    resultaat = json.load(f)

nieuw = resultaat["content"]
oud   = parse_content(src_by_id[INSPECT][cols["content"]])
print("sleutels gelijk aan de bron:", set(nieuw) == set(oud))

for kopje in sjabloon.KOPJES:
    v = nieuw.get(kopje.cms, "")
    print(f"\n{'='*70}\n{kopje.kop}  ({kopje.cms})\n{'='*70}")
    print(v if isinstance(v, str) else repr(v))

sleutels gelijk aan de bron: True

Overzicht  (summary)
Wil je zelfstandig webapplicaties bouwen met PHP en MySQL? Tijdens deze opleiding leer je programmeren van functioneel naar objectgeoriënteerd, met security als vaste rode draad. Je werkt met Composer, PSR-standaarden en moderne taalfeatures, en bouwt een eigen webapplicatie zoals een webwinkel. Na afloop schrijf je onderhoudbare code, zet je een veilige database op en koppel je die aan je applicatie.

Inleiding  (intro)
<p>PHP draait achter een groot deel van het web, van maatwerkapplicaties tot bekende platformen. Tijdens deze opleiding bouw je in vijf dagen een stevige basis op als PHP-ontwikkelaar. Je begint met functioneel programmeren: variabelen, arrays, lussen, functies en het afhandelen van fouten. Daarna stap je over naar objectgeoriënteerd werken, waarbij je zowel externe als eigen classes inzet en kennismaakt met design patterns.</p>
<p>Databases vormen het tweede fundament. Je zet zelf een MySQL-database op, bevraagt 

## 8. Eén kopje bijsturen

Is er één kopje mis, dan hoef je de hele training niet opnieuw te genereren. Zet `KOPJE` op het
veld dat je wilt hergenereren en geef optioneel een `COMMENT` mee met wat er anders moet.
Zonder comment is het een gewone retry.

De schrijver krijgt de volledige huidige training als context, zodat het nieuwe kopje aansluit
op wat er al staat. De per-training-artefacten (`<id>.json` én `<id>.md`) en de rij in
`herschreven.xlsx` worden bijgewerkt.

In [ ]:
KOPJE   = "modules"    # kies uit rw.HERGENEREERBAAR
COMMENT = ""           # bv. "module 2 en 4 overlappen; voeg ze samen en voeg een module over X toe"

print("hergenereerbaar:", ", ".join(rw.HERGENEREERBAAR), "\n")

with open(f"{OUT_DIR}/trainingen/{INSPECT}.json", encoding="utf-8") as f:
    voor = json.load(f)
oud = rw._writer_out_uit_json(voor).get(KOPJE)

res = rw.hergenereer_kopje_op_schijf(SCORED, SOURCE, INSPECT, KOPJE, COMMENT,
                                     besluiten_path=BESLUITEN, out_dir=OUT_DIR)

print(f"\n{'='*70}\nOUD\n{'='*70}")
print(json.dumps(oud, ensure_ascii=False, indent=2)[:2000])
print(f"\n{'='*70}\nNIEUW\n{'='*70}")
print(json.dumps(res.writer_out.get(KOPJE), ensure_ascii=False, indent=2)[:2000])

## 10. Goud-corpus: exporteren en kalibreren

De trainingen die al in de nieuwe stijl staan (`herschreven=1`) worden **niet** herschreven —
ze gaan het sheet in op modus `overnemen` (status `overgenomen`). Als corpus hebben ze twee
functies.

`checks_over_goud()` draait de code-check over alle 78 en telt per regel hoe vaak die omvalt.
Dat is een meetlat in twee richtingen:

- **valt een regel bij meer dan de helft om**, dan is de regel verdacht en niet de training —
  zo viel het oude harde lengtevenster om, en is de lengte nu een richtlijn met een vangrail
  (`lengtes_over_goud()` meet de verdeling waarop die banden zijn gekozen);
- **de trainingen die álles halen** zijn de few-shot-voorbeelden. Dat zijn er 7; vier ervan
  staan in `rw.GOUD_VOORBEELDEN` en gaan mee in de gecachete system-prefix van de schrijver.

**De modules tellen sinds kort mee.** Zolang `goud_naar_check_input` de modulestructuur
oversloeg waren `modules_aantal`, `bullets_aantal` en `bullets_variatie` op het goud
onzichtbaar. Met een diepteteller in plaats van een regex is die structuur wél te lezen (75 van
de 78), en zakte het aantal schone trainingen van 26 naar 7 — drie van de vier oude voorbeelden
demonstreerden modules met twee sub-bullets terwijl de spec er 3–6 eist.

Let daarbij op `bullets_aantal`: dat valt 172 keer om over 78 trainingen. Lees dat als een vraag
aan de spec, niet als een oordeel over het goud — precies waar de eerste regel hierboven voor is.

Verander je een check, draai deze cel dan opnieuw en werk `GOUD_VOORBEELDEN` bij.

In [ ]:
rw.export_goud_corpus(SOURCE, OUT_DIR)
print()
meting = rw.checks_over_goud()

schoon = {t for t, _ in meting["schoon"]}
print(f"\nnu vastgelegd als few-shot: {rw.GOUD_VOORBEELDEN}")
# de few-shot moet een DEELVERZAMELING zijn van wat alles haalt, niet gelijk eraan --
# anders demonstreert een voorbeeld de regel die het hoort te tonen niet
vervuild = [t for t in rw.GOUD_VOORBEELDEN if t not in schoon]
if vervuild:
    print(f"LET OP: {vervuild} haalt niet meer alle harde checks -> werk GOUD_VOORBEELDEN bij "
          f"in rewrite_trainings.py. Kandidaten: {sorted(schoon)}")
else:
    print(f"alle vier halen alle harde checks ({len(schoon)} kandidaten in totaal)")

## 10. Goud-corpus: exporteren en kalibreren

De trainingen die al in de nieuwe stijl staan (`herschreven=1`) worden **niet** herschreven —
ze gaan ongewijzigd het sheet in (status `overgenomen`). Als corpus hebben ze twee functies.

`checks_over_goud()` draait de code-check over alle 78 en telt per regel hoe vaak die omvalt.
Dat is een meetlat in twee richtingen:

- **valt een regel bij meer dan de helft om**, dan is de regel verdacht en niet de training —
  zo viel het oude harde lengtevenster om, en is de lengte nu een richtlijn met een vangrail
  (`lengtes_over_goud()` meet de verdeling waarop die banden zijn gekozen);
- **de trainingen die álles halen** zijn de few-shot-voorbeelden. Dat zijn er 23; vier ervan
  staan in `rw.GOUD_VOORBEELDEN` en gaan mee in de gecachete system-prefix van de schrijver.

Verander je een check, draai deze cel dan opnieuw en werk `GOUD_VOORBEELDEN` bij.

In [ ]:
rw.export_goud_corpus(SOURCE, OUT_DIR)
print()
meting = rw.checks_over_goud()

print(f"\nnu vastgelegd als few-shot: {rw.GOUD_VOORBEELDEN}")
if tuple(t for t, _ in meting["schoon"]) != rw.GOUD_VOORBEELDEN:
    print("LET OP: wijkt af van de meting -> werk GOUD_VOORBEELDEN bij in rewrite_trainings.py")